In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

sys.path.append('../src')

from data_pipeline import load_all_raw_data

from data_analysis import (
    filter_data_until_date, temporal_split_data, 
)

from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)

from preprocess import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)

from weather_data import WeatherDataCollector


In [3]:
trips_with_weather = pd.read_csv('../data/processed/trips_with_weather.csv')

In [ ]:
# Convert trips_with_weather to parquet format for more efficient storage and faster loading
trips_with_weather.to_parquet('../data/processed/trips_with_weather.parquet', compression='snappy')


In [ ]:
trips_with_weather = pd.read_parquet('../data/processed/trips_with_weather.parquet')

In [4]:
trips_with_weather.columns

Index(['id_recorrido', 'duracion_recorrido', 'fecha_origen_recorrido',
       'id_estacion_origen', 'nombre_estacion_origen',
       'direccion_estacion_origen', 'long_estacion_origen',
       'lat_estacion_origen', 'fecha_destino_recorrido', 'id_estacion_destino',
       'nombre_estacion_destino', 'direccion_estacion_destino',
       'long_estacion_destino', 'lat_estacion_destino', 'id_usuario',
       'modelo_bicicleta', 'genero', 'weather_temperature_2m',
       'weather_relative_humidity_2m', 'weather_dew_point_2m',
       'weather_apparent_temperature', 'weather_precipitation', 'weather_rain',
       'weather_weather_code', 'weather_pressure_msl',
       'weather_surface_pressure', 'weather_cloud_cover',
       'weather_cloud_cover_low', 'weather_cloud_cover_mid',
       'weather_cloud_cover_high', 'weather_et0_fao_evapotranspiration',
       'weather_vapour_pressure_deficit', 'weather_wind_speed_10m',
       'weather_wind_speed_100m', 'weather_wind_direction_10m',
       'weather

: 

In [ ]:
from data_analysis import engineer_ecobici_features

df_feat = engineer_ecobici_features(trips_with_weather)

df_feat.describe()

🚴‍♂️  Parametrized Feature Engineering Pipeline (11 steps)
  Checkpoints directory: 'feature_checkpoints/'
  Input raw data shape: (12976053, 48)
  -> Memory usage after initialization: 3308.9 MB
[Step 1/11] Converting to Polars and normalizing dates...
  -> Loading Polars conversion from checkpoint: step01_polars_conversion.parquet
[Step 2/11] Creating departures table with lags & user profiles...
  -> Loading departures table from checkpoint: step02_departures.parquet
[Step 3/11] Creating arrivals table with lags...
  -> Loading arrivals table from checkpoint: step03_arrivals.parquet
[Step 4/11] Creating shifted arrivals table for target...
  -> Loading shifted arrivals from checkpoint: step04_shifted_arrivals.parquet
[Step 5/11] Creating enhanced weather table...
  -> Loading weather table from checkpoint: step05_weather.parquet
[Step 6/11] Creating station metadata table...
  -> Loading station metadata from checkpoint: step06_stations.parquet
[Step 7/11] Building full station x ti

Sorting data:  20%|██        | 1/5 [00:00<00:00, 250.11it/s]

In [ ]:
import polars as pl

In [ ]:
df_feat = pl.read_parquet(r"C:\Users\xxx\Documents\GitHub\EcoBici-AI\notebooks\feature_checkpoints\df_feat_final.parquet")

In [ ]:
df_feat.columns

['station_id',
 'ts_start',
 'dep_last_DT',
 'share_male',
 'dep_lag_1',
 'dep_lag_2',
 'dep_lag_3',
 'dep_lag_4',
 'dep_lag_5',
 'dep_lag_6',
 'y_arrivals_next_DT',
 'weather_temperature_2m',
 'weather_relative_humidity_2m',
 'weather_dew_point_2m',
 'hour',
 'dow',
 'month',
 'sin_hour',
 'cos_hour',
 'is_weekend',
 'near_dep_sum_DT']

In [ ]:
df_feat.sample(5)

station_id,ts_start,dep_last_DT,share_male,dep_lag_1,dep_lag_2,dep_lag_3,dep_lag_4,dep_lag_5,dep_lag_6,y_arrivals_next_DT,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,hour,dow,month,sin_hour,cos_hour,is_weekend,near_dep_sum_DT
i64,datetime[μs],u32,f64,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,i8,i8,i8,f64,f64,i8,u32
64,2021-03-21 19:30:00,0,null,0,0,0,0,0,0,2,22.411001,57.87981,13.711,19,7,3,-0.965926,0.258819,1,2
377,2023-10-05 04:30:00,0,null,0,0,0,0,0,0,0,9.261001,70.4495,4.161,4,4,10,0.866025,0.5,0,0
235,2023-07-17 09:00:00,0,null,0,0,0,0,0,0,1,2.211,83.23418,-0.339,9,1,7,0.707107,-0.707107,0,2
524,2024-03-26 23:30:00,1,0.0,3,1,1,3,1,2,0,19.811,83.37646,16.911001,23,2,3,-0.258819,0.965926,0,1
306,2020-03-10 04:30:00,0,null,0,0,0,0,0,0,0,20.461,85.0439,17.861,4,2,3,0.866025,0.5,0,0


In [ ]:
# Check if GPU is available
import xgboost as xgb
print("\nChecking GPU availability for XGBoost...")
try:
    # Build a small test DMatrix
    import numpy as np
    test_data = np.random.rand(10, 10)
    test_label = np.random.rand(10)
    test_dmatrix = xgb.DMatrix(test_data, label=test_label)
    
    # Try to train a small model with GPU
    test_params = {'tree_method': 'gpu_hist'}
    test_bst = xgb.train(test_params, test_dmatrix, num_boost_round=1)
    print("✓ GPU is available and working")
    use_gpu = True
except Exception as e:
    print("✗ GPU not available:", str(e))
    print("Falling back to CPU training")
    use_gpu = False

# Split data temporally for training and validation
print("\nSplitting data temporally...")
split_date = pl.datetime(2023, 9, 1)  # Fixed datetime constructor call
train_data = df_feat.filter(pl.col("ts_start") < split_date)
val_data = df_feat.filter(pl.col("ts_start") >= split_date)

# Define features and target
target_col = 'y_arrivals_next_DT'
# Train XGBoost model with chunking
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set XGBoost parameters
params = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
}

# Add GPU parameters if available
if use_gpu:
    params.update({
        'tree_method': 'gpu_hist',
        'predictor': 'gpu_predictor',
        'gpu_id': 0
    })
else:
    params.update({
        'tree_method': 'hist'
    })

# Initialize model
model = xgb.XGBRegressor(**params)

# Process data in chunks
chunk_size = 100000  # Adjust based on available memory

print(f"\nTraining XGBoost model in chunks using {'GPU' if use_gpu else 'CPU'}...")
for i in range(0, len(train_data), chunk_size):
    # Get chunk of data
    chunk_end = min(i + chunk_size, len(train_data))
    print(f"Processing chunk {i//chunk_size + 1}, rows {i} to {chunk_end}")
    
    # Convert chunk to pandas
    train_chunk = train_data[i:chunk_end].to_pandas()
    
    # Update model with this chunk
    model.fit(
        train_chunk[feature_cols],
        train_chunk[target_col],
        xgb_model=model.get_booster() if i > 0 else None
    )

print("\nMaking predictions...")
# Process validation data in chunks too
val_pred = []
val_true = []
for i in range(0, len(val_data), chunk_size):
    chunk_end = min(i + chunk_size, len(val_data))
    val_chunk = val_data[i:chunk_end].to_pandas()
    
    chunk_pred = model.predict(val_chunk[feature_cols])
    val_pred.extend(chunk_pred)
    val_true.extend(val_chunk[target_col].values)

# Evaluate model
print("\nValidation metrics:")
print(f"MSE: {mean_squared_error(val_true, val_pred):.4f}")
print(f"MAE: {mean_absolute_error(val_true, val_pred):.4f}")
print(f"R2: {r2_score(val_true, val_pred):.4f}")

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 most important features:")
print(importance.head(10))

# Save model
model.save_model('xgb_model.json')
print("\nModel saved as 'xgb_model.json'")



Checking GPU availability for XGBoost...
✓ GPU is available and working

Splitting data temporally...

Training XGBoost model in chunks using GPU...
Processing chunk 1, rows 0 to 100000
Processing chunk 2, rows 100000 to 200000
Processing chunk 3, rows 200000 to 300000
Processing chunk 4, rows 300000 to 400000
Processing chunk 5, rows 400000 to 500000
Processing chunk 6, rows 500000 to 600000
Processing chunk 7, rows 600000 to 700000
Processing chunk 8, rows 700000 to 800000
Processing chunk 9, rows 800000 to 900000
Processing chunk 10, rows 900000 to 1000000
Processing chunk 11, rows 1000000 to 1100000
Processing chunk 12, rows 1100000 to 1200000
Processing chunk 13, rows 1200000 to 1300000
Processing chunk 14, rows 1300000 to 1400000
Processing chunk 15, rows 1400000 to 1500000
Processing chunk 16, rows 1500000 to 1600000
Processing chunk 17, rows 1600000 to 1700000
Processing chunk 18, rows 1700000 to 1800000
Processing chunk 19, rows 1800000 to 1900000
Processing chunk 20, rows 19